# Explore fast routing and path saving

In [ ]:
import pickle
import sqlite3
from collections import defaultdict

import h5py
import igraph
import networkx
import osmnx
import rocksdb
from tqdm.notebook import tqdm

In [ ]:
G = osmnx.graph_from_point((37.79, -122.41), dist=5000, network_type="all_private")

In [ ]:
nodes, edges = osmnx.graph_to_gdfs(G)

In [ ]:
len(nodes), len(edges) 

In [ ]:
nodes.plot(markersize=0.1)

In [ ]:
# check what filter is used
# osmnx.downloader._get_osm_filter('all_private')

In [ ]:
def list_to_str(e):
    if type(e) is list:
        return ",".join(str(i) for i in e)
    else:
        return str(e)
edges.osmid = edges.osmid.apply(list_to_str)
edges.highway = edges.highway.apply(list_to_str)
edges = edges[['osmid', 'highway', 'geometry']]

In [ ]:
nodes.to_parquet('nodes.gpq')
edges.to_parquet('edges.gpq')

In [ ]:
components = [c for c in networkx.connected_components(G.to_undirected())]
largest_component = max(components, key=len)
H = G.to_undirected().subgraph(largest_component)
len(G), len(H), len(components)

In [ ]:
h = igraph.Graph.from_networkx(H)

In [ ]:
sources = nodes.sample(50)
targets = nodes.sample(500)

In [ ]:
def edge_ids(nodes):
    next_nodes = iter(nodes)
    next(next_nodes)
    return set((e[0], e[1], 0) for e in zip(nodes, next_nodes))

pair_to_path = {}
edge_to_pairs = defaultdict(list)

for u in tqdm(sources.index, desc="sources", total=len(sources)):
    paths = networkx.shortest_path(H, source=u)
    for v in targets.index:
        path = paths[v]
        edge_path = edge_ids(path)
        pair_to_path[(u,v)] = path
        for edge in edge_path:
            edge_to_pairs[edge].append((u,v))

In [ ]:
edge_to_pairs[(65283514, 65333602, 0)]

In [ ]:
sources.index

In [ ]:
nodes_ig = nodes.reset_index()
edges_ig = edges.reset_index()
sources_ig = nodes_ig[nodes_ig.osmid.isin(sources.index)]
targets_ig = nodes_ig[nodes_ig.osmid.isin(targets.index)]
targets_ig.head(1)

In [ ]:
h.vs[0]

In [ ]:
edge_to_pairs_ig = defaultdict(list)

for u_ig in tqdm(sources_ig.index, desc="sources", total=len(sources)):
    u = h.vs[u_ig]['_nx_name']
    paths = h.get_shortest_paths(u_ig, targets_ig.index, output="vpath")
    for path_i, v_ig in enumerate(targets_ig.index):
        v = h.vs[v_ig]['_nx_name']
        nodes = [h.vs[n]['_nx_name'] for n in paths[path_i]]
        next_nodes = iter(nodes)
        next(next_nodes)
        for a, b in zip(nodes, next_nodes):
            edge_to_pairs_ig[(a,b,0)].append((u, v))

In [ ]:
set(edge_to_pairs_ig[(65283514, 65333602, 0)]) == set(edge_to_pairs[(65283514, 65333602, 0)])

In [ ]:
i = 0
for edge in edge_to_pairs:
    if not set(edge_to_pairs[edge]) == set(edge_to_pairs_ig[edge]):
        print(edge)
        i = i + 1
        break
i, len(edge_to_pairs)

In [ ]:
set(edge_to_pairs_ig[edge]) - set(edge_to_pairs[edge])

## SQLite

Write to a single file, gives us full database functionality to store things in tables, create indexes.

Nice thing here is that with a table format we can do append only for pair

In [ ]:
con = sqlite3.connect("paths.sqlite")
cur = con.cursor()
#cur.execute("drop table if exists pair_to_path")
cur.execute("drop table if exists edge_to_pair")
#cur.execute("create table pair_to_path (u text, v text, path text)")
cur.execute("create table edge_to_pair (u text, v text, a text, b text)")
con.commit()
con.close()

In [ ]:
con = sqlite3.connect("paths.sqlite")
cur = con.cursor()

for u in tqdm(sources.index, desc="sources", total=len(sources)):
    paths = networkx.shortest_path(H, source=u)
    for v in targets.index:
        try:
            path = paths[v]
            edge_path = edge_ids(path)
            #cur.execute("insert into pair_to_path values ('%s','%s','%s')" % (u, v, path))
            for a, b, _ in edge_path:
                cur.execute("insert into edge_to_pair values ('%s','%s','%s','%s')" % (u, v, a, b))
        except KeyError:
            print(u,v)

con.commit()
con.close()

In [ ]:
con = sqlite3.connect("paths.sqlite")
cur = con.cursor()
# rows = cur.execute("select path from pair_to_path where u = '%s' and v = '%s'" % (6318851267, 4548575761))
# for r in rows:
#     print(r)

rows = cur.execute("select u, v from edge_to_pair where a = '%s' and b = '%s'" % (65283514, 65333602))
for r in rows:
    print(r)

con.commit()
con.close()

# HDF5

Attempt writing to HDF5 dataset - this may not be a good fit as we do lots of read-and-check. Seems extremely slow.

In [ ]:
with h5py.File("edge_to_pair.hdf5", 'w') as f:
    for u in tqdm(sources.index, desc="sources", total=len(sources)):
        paths = networkx.shortest_path(H, source=u)
        for v in targets.index:
            try:
                path = paths[v]
                edge_path = [str(e) for e in edge_ids(path)]
                for edge in edge_path:
                    #edge_to_pair[edge].append((u,v))
                    if edge in f:
                        pairs = list(f[edge])
                        pairs.append((u,v))
                        del f[edge]
                    else:
                        pairs = [(u,v)]
                    f[edge] = pairs
            except KeyError:
                print(u,v)
        break

# RocksDB

On-disk key-value store with a "MergeOperator" interface that looks promising, so we can append, but was killing the kernel when testing. 

Downside is that keys and values must be bytes, so we need to serialise anyway - tried using pickle, possibly slightly faster using ascii-encoded bytes string. Other options?

In [ ]:
# class ListAppend(rocksdb.interfaces.AssociativeMergeOperator):
#     def merge(self, key, existing_value, value):
#         if existing_value:
#             s = f"{existing_value},{value}"
#             return (True, str(s).encode('ascii'))
#         return (True, value)

#     def name(self):
#         return b'ListAppend'
    
opts = rocksdb.Options()
opts.create_if_missing = True
# opts.merge_operator = ListAppend()

# db is closed when object goes out of scope
# this try/except forces it to close before we try opening it again
try:
    del db
except NameError:
    pass

db = rocksdb.DB("edge_to_pair.db", opts)

# careful to delete from db if re-running this
keys = db.iterkeys()
keys.seek_to_first()
for key in keys:
    db.delete(key)

for u in tqdm(sources.index, desc="sources", total=len(sources)):
    paths = networkx.shortest_path(H, source=u)
#     batch = rocksdb.WriteBatch()
    for v in targets.index:
        try:
            path = paths[v]
            edge_path = edge_ids(path)
            for edge in edge_path:
#                 batch.merge(edge, str((u,v)).encode('ascii'))
                pair = (u,v)
                #edge_p = pickle.dumps(edge)
                edge_p = str(edge).encode('ascii')
                pairs_p = db.get(edge_p)
                if pairs_p is None:
                    #pairs = [pair]
                    pairs_p = str(pair).encode('ascii')
                else:
                    #pairs = pickle.loads(pairs_p)
                    #pairs.append(pair)
                    pairs_p = pairs_p + b"|" + str(pair).encode('ascii')
                #pairs_p = pickle.dumps(pairs)
                db.put(edge_p, pairs_p)
        except KeyError:
            print(u,v)
#     db.write(batch)
            
del db

In [ ]:
del opts

In [ ]:
try:
    del db
except NameError:
    pass
db = rocksdb.DB("edge_to_pair.db", rocksdb.Options(create_if_missing=True))
e = (65283514, 65333602, 0)
#pairs = db.get(pickle.dumps(e))
pairs = db.get(str(e).encode('ascii'))
del db
#pickle.loads(pairs)
# massive hack to "deserialize" from |-separated tuple string
[eval(p.decode('ascii')) for p in pairs.split(b"|")]

In [ ]:
H.edges[(path[0],path[1],0)]

In [ ]:
edges.loc[32927563,645559609,0]

In [ ]:
edges.iloc[1]